In [40]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import re
import ast
from collections import Counter

In [41]:
# File paths
telemetry_dir_name = 'telemetry20251014_171334'

flow_stats_path = f"../{telemetry_dir_name}/flow_stats.csv"
port_stats_path = f"../{telemetry_dir_name}/port_stats.csv"
table_stats_path = f"../{telemetry_dir_name}/table_stats.csv"
alerts_path = f"../{telemetry_dir_name}/alerts.csv"

# Filter data to start - end range - taken from simulation log.
start_time = pd.to_datetime('2025-10-14 19:14:18')
end_time = pd.to_datetime('2025-10-14 20:15:09')

output_path = "ml_dataset_50-50_14-10-2025.csv"

In [42]:
def parse_alert_timestamp(alert_ts):
    """Convert alert timestamp format '10/12-14:03:19.850126' to datetime"""
    # Handle NaN
    if pd.isna(alert_ts) or not isinstance(alert_ts, str):
        return pd.NaT
    
    try:
        # Format: '10/12-14:03:19.850126'
        month_day, time_part = alert_ts.split('-')
        month, day = month_day.split('/')
        
        dt_str = f"2025-{month.zfill(2)}-{day.zfill(2)} {time_part}"
        return pd.to_datetime(dt_str, format='%Y-%m-%d %H:%M:%S.%f')
    except Exception as e:
        print(f"Warning: Could not parse alert timestamp '{alert_ts}': {e}")
        return pd.NaT

def parse_stats_timestamp(ts):
    """Parse stats timestamp format 'Sun Oct 12 13:54:29 2025'"""
    if pd.isna(ts):
        return pd.NaT
    try:
        # Format: 'Sun Oct 12 13:54:29 2025'
        return pd.to_datetime(ts, format='%a %b %d %H:%M:%S %Y')
    except Exception as e:
        print(f"Warning: Could not parse stats timestamp '{ts}': {e}")
        return pd.NaT

def extract_match_fields(match_str):
    """Extract fields from OFPMatch string"""
    if pd.isna(match_str) or match_str == '':
        return {}
    
    try:
        # Extract the dictionary part from OFPMatch string
        match = re.search(r'oxm_fields=({.*})', str(match_str))
        if match:
            dict_str = match.group(1)
            return ast.literal_eval(dict_str)
    except:
        pass
    return {}

def get_protocol_from_match(match_dict):
    """Determine protocol type from match fields"""
    if 'ip_proto' in match_dict:
        proto = match_dict['ip_proto']
        if proto == 6:
            return 'tcp'
        elif proto == 17:
            return 'udp'
        elif proto == 1:
            return 'icmp'
    return 'other'

def aggregate_features(df_group, window_stats=None):
    """Aggregate features for a time window"""
    features = {}
    
    # Basic aggregations
    numeric_cols = df_group.select_dtypes(include=[np.number]).columns
    
    for col in numeric_cols:
        if col in df_group.columns:
            features[f'{col}_sum'] = df_group[col].sum()
            features[f'{col}_mean'] = df_group[col].mean()
            features[f'{col}_std'] = df_group[col].std()
            features[f'{col}_min'] = df_group[col].min()
            features[f'{col}_max'] = df_group[col].max()
    
    # Add window statistics if available
    if window_stats is not None:
        for key, value in window_stats.items():
            features[key] = value
    
    return features

In [43]:
def create_ml_dataset(flow_stats_path, port_stats_path, table_stats_path, alerts_path):
    """Create ML dataset from SDN statistics and Snort alerts"""

    # Load datasets
    flow_df = pd.read_csv(flow_stats_path)
    port_df = pd.read_csv(port_stats_path)
    table_df = pd.read_csv(table_stats_path)
    
    # Load alerts - the timestamp is the first column
    alerts_df = pd.read_csv(alerts_path, sep='\s+', usecols=[0], names=['alert_timestamp'], header=None)

    # Parse timestamps
    print("Parsing timestamps...")
    flow_df['timestamp'] = flow_df['ts'].apply(parse_stats_timestamp)
    port_df['timestamp'] = port_df['ts'].apply(parse_stats_timestamp)
    table_df['timestamp'] = table_df['ts'].apply(parse_stats_timestamp)
    
    alerts_df['timestamp'] = alerts_df['alert_timestamp'].apply(parse_alert_timestamp)
    
    # Remove NaT values from alerts
    print(f"Alerts before removing NaT: {len(alerts_df)}")
    alerts_df = alerts_df[alerts_df['timestamp'].notna()]
    print(f"Alerts after removing NaT: {len(alerts_df)}")

    print(f"\nFiltering data to time range: {start_time} to {end_time}")
    flow_df = flow_df[(flow_df['timestamp'] >= start_time) & (flow_df['timestamp'] <= end_time)]
    port_df = port_df[(port_df['timestamp'] >= start_time) & (port_df['timestamp'] <= end_time)]
    table_df = table_df[(table_df['timestamp'] >= start_time) & (table_df['timestamp'] <= end_time)]
    alerts_df = alerts_df[(alerts_df['timestamp'] >= start_time) & (alerts_df['timestamp'] <= end_time)]
    
    print(f"Rows after filtering - Flow: {len(flow_df)}, Port: {len(port_df)}, Table: {len(table_df)}, Alerts: {len(alerts_df)}")
    
    # Round timestamps to seconds for aggregation
    flow_df['time_bucket'] = flow_df['timestamp'].dt.floor('1s')
    port_df['time_bucket'] = port_df['timestamp'].dt.floor('1s')
    table_df['time_bucket'] = table_df['timestamp'].dt.floor('1s')
    alerts_df['time_bucket'] = alerts_df['timestamp'].dt.floor('1s')
    
    # Create attack indicator
    attack_times = set(alerts_df['time_bucket'])
    
    # Debug: Print sample timestamps
    if len(flow_df) > 0:
        print(f"\nSample flow timestamp: {flow_df['timestamp'].iloc[0]}")
        print(f"Sample flow time_bucket: {flow_df['time_bucket'].iloc[0]}")
    if len(alerts_df) > 0:
        print(f"Sample alert timestamp: {alerts_df['timestamp'].iloc[0]}")
        print(f"Sample alert time_bucket: {alerts_df['time_bucket'].iloc[0]}")
    print(f"Total alerts: {len(alerts_df)}")
    
    # Debug: Check if any timestamps match
    print(f"\nChecking timestamp overlap...")
    print(f"Flow time range: {flow_df['time_bucket'].min()} to {flow_df['time_bucket'].max()}")
    print(f"Alert time range: {alerts_df['time_bucket'].min() if len(alerts_df) > 0 else 'N/A'} to {alerts_df['time_bucket'].max() if len(alerts_df) > 0 else 'N/A'}")
    
    # Show first few alert time buckets
    if len(alerts_df) > 0:
        print(f"\nFirst 5 alert time buckets:")
        print(alerts_df['time_bucket'].head().tolist())
        print(f"\nFirst 5 flow time buckets:")
        print(flow_df['time_bucket'].head().tolist())
    
    # Extract match fields from flow stats
    print("Extracting flow match fields...")
    flow_df['match_dict'] = flow_df['match'].apply(extract_match_fields)
    flow_df['protocol'] = flow_df['match_dict'].apply(get_protocol_from_match)
    flow_df['src_ip'] = flow_df['match_dict'].apply(lambda x: x.get('ipv4_src', None))
    flow_df['dst_ip'] = flow_df['match_dict'].apply(lambda x: x.get('ipv4_dst', None))
    flow_df['src_port'] = flow_df['match_dict'].apply(lambda x: x.get('tcp_src', x.get('udp_src', None)))
    flow_df['dst_port'] = flow_df['match_dict'].apply(lambda x: x.get('tcp_dst', x.get('udp_dst', None)))
    
    # Get all unique time buckets
    all_times = sorted(set(flow_df['time_bucket']) | set(port_df['time_bucket']) | set(table_df['time_bucket']))
    
    print(f"Processing {len(all_times)} time windows...")
    
    dataset_rows = []
    
    for i, time_bucket in enumerate(all_times):
        if i % 100 == 0:
            print(f"Processing {i}/{len(all_times)}...")
        
        row = {'timestamp': time_bucket}
        
        # Get data for current time bucket
        flow_window = flow_df[flow_df['time_bucket'] == time_bucket]
        port_window = port_df[port_df['time_bucket'] == time_bucket]
        table_window = table_df[table_df['time_bucket'] == time_bucket]
                
        # Flow statistics aggregation
        if len(flow_window) > 0:
            row['flow_packet_count_sum'] = flow_window['packet_count'].sum()
            row['flow_packet_count_mean'] = flow_window['packet_count'].mean()
            row['flow_packet_count_std'] = flow_window['packet_count'].std()
            
            row['flow_byte_count_sum'] = flow_window['byte_count'].sum()
            row['flow_byte_count_mean'] = flow_window['byte_count'].mean()
            row['flow_byte_count_std'] = flow_window['byte_count'].std()
            
            row['flow_duration_sec_mean'] = flow_window['duration_sec'].mean()
            row['flow_duration_sec_std'] = flow_window['duration_sec'].std()
        else:
            row['flow_packet_count_sum'] = 0
            row['flow_packet_count_mean'] = 0
            row['flow_packet_count_std'] = 0
            
            row['flow_byte_count_sum'] = 0
            row['flow_byte_count_mean'] = 0
            row['flow_byte_count_std'] = 0
            
            row['flow_duration_sec_mean'] = 0
            row['flow_duration_sec_std'] = 0
        
        # Port statistics aggregation
        if len(port_window) > 0:
            row['port_rx_packets_sum'] = port_window['rx_packets'].sum()
            row['port_rx_packets_mean'] = port_window['rx_packets'].mean()
            row['port_rx_packets_std'] = port_window['rx_packets'].std()
            
            row['port_tx_packets_sum'] = port_window['tx_packets'].sum()
            row['port_tx_packets_mean'] = port_window['tx_packets'].mean()
            row['port_tx_packets_std'] = port_window['tx_packets'].std()
            
            row['port_rx_bytes_sum'] = port_window['rx_bytes'].sum()
            row['port_rx_bytes_mean'] = port_window['rx_bytes'].mean()
            row['port_rx_bytes_std'] = port_window['rx_bytes'].std()
            
            row['port_tx_bytes_sum'] = port_window['tx_bytes'].sum()
            row['port_tx_bytes_mean'] = port_window['tx_bytes'].mean()
            row['port_tx_bytes_std'] = port_window['tx_bytes'].std()
            
            row['port_rx_dropped_sum'] = port_window['rx_dropped'].sum()
            row['port_rx_dropped_mean'] = port_window['rx_dropped'].mean()
            row['port_rx_dropped_std'] = port_window['tx_dropped'].sum()
            
            row['port_tx_errors_sum'] = port_window['tx_errors'].sum()
            row['port_tx_errors_mean'] = port_window['tx_errors'].mean()
            row['port_tx_errors_std'] = port_window['tx_errors'].std()
        else:
            for metric in ['rx_packets', 'tx_packets', 'rx_bytes', 'tx_bytes', 'rx_dropped', 'tx_dropped', 'rx_errors', 'tx_errors']:
                row[f'port_{metric}_sum'] = 0
                if metric in ['rx_packets', 'tx_packets', 'rx_bytes', 'tx_bytes']:
                    row[f'port_{metric}_mean'] = 0
                    row[f'port_{metric}_std'] = 0
        
        # Table statistics aggregation
        if len(table_window) > 0:
            row['table_active_count_sum'] = table_window['active_count'].sum()
            row['table_active_count_mean'] = table_window['active_count'].mean()
            row['table_active_count_std'] = table_window['active_count'].std()

            row['table_lookup_count_sum'] = table_window['lookup_count'].sum()
            row['table_lookup_count_mean'] = table_window['lookup_count'].mean()
            row['table_lookup_count_std'] = table_window['lookup_count'].std()
            
            row['table_matched_count_sum'] = table_window['matched_count'].sum()
            row['table_matched_count_mean'] = table_window['matched_count'].mean()
            row['table_matched_count_std'] = table_window['matched_count'].std()
        else:
            for metric in ['active_count', 'lookup_count', 'matched_count']:
                row[f'table_{metric}_sum'] = 0
                row[f'table_{metric}_mean'] = 0
                row[f'table_{metric}_std'] = 0

        # Protocol flow counts
        protocol_counts = flow_window['protocol'].value_counts()
        row['no_of_tcp_flows'] = protocol_counts.get('tcp', 0)
        row['no_of_udp_flows'] = protocol_counts.get('udp', 0)
        row['no_of_icmp_flows'] = protocol_counts.get('icmp', 0)
        row['no_of_other_flows'] = protocol_counts.get('other', 0)
        row['no_of_total_flows'] = len(flow_window)

        # Unique source/destination addresses
        row['no_of_unique_src_addresses'] = flow_window['src_ip'].dropna().nunique()
        row['no_of_unique_dst_addresses'] = flow_window['dst_ip'].dropna().nunique()
        
        # Unique source/destination ports
        row['no_of_unique_src_ports'] = flow_window['src_port'].dropna().nunique()
        row['no_of_unique_dst_ports'] = flow_window['dst_port'].dropna().nunique()

        # Flow ratios
        row['tcp_to_udp_ratio'] = row['no_of_tcp_flows'] / (row['no_of_udp_flows'] + 1)
        row['flow_packets_per_flow'] = row['flow_packet_count_sum'] / (row['no_of_total_flows'] + 1)

        # Asymmetry ratios
        row['unique_src_to_dst_address_ratio'] = row['no_of_unique_src_addresses'] / (row['no_of_unique_dst_addresses'] + 1)
        row['unique_src_to_dst_port_ratio'] = row['no_of_unique_src_ports'] / (row['no_of_unique_dst_ports'] + 1)
        
        # Last 10 seconds statistics
        time_10s_ago = time_bucket - timedelta(seconds=10)
        flow_10s = flow_df[(flow_df['time_bucket'] > time_10s_ago) & (flow_df['time_bucket'] <= time_bucket)]
        port_10s = port_df[(port_df['time_bucket'] > time_10s_ago) & (port_df['time_bucket'] <= time_bucket)]
        
        row['flow_packet_count_10s_mean'] = flow_10s['packet_count'].mean() if len(flow_10s) > 0 else 0
        row['flow_byte_count_10s_mean'] = flow_10s['byte_count'].mean() if len(flow_10s) > 0 else 0
        row['port_rx_packets_10s_mean'] = port_10s['rx_packets'].mean() if len(port_10s) > 0 else 0
        row['port_tx_packets_10s_mean'] = port_10s['tx_packets'].mean() if len(port_10s) > 0 else 0
        row['total_flows_10s'] = len(flow_10s)
        
        # Attack label
        row['attack'] = 1 if time_bucket in attack_times else 0
        
        dataset_rows.append(row)
    
    # Create DataFrame
    print("Creating final dataset...")
    dataset = pd.DataFrame(dataset_rows)
    
    # Calculate differences from previous timestamp
    print("Calculating temporal differences...")
    diff_columns = [
        'flow_packet_count_sum', 
        'flow_byte_count_sum',
        'port_rx_packets_sum', 
        'port_tx_packets_sum',
        'port_rx_bytes_sum', 
        'port_tx_bytes_sum',
        'no_of_total_flows', 
        'no_of_tcp_flows',
        'no_of_udp_flows',
        'no_of_icmp_flows',
        'no_of_other_flows',
        'no_of_unique_src_addresses',
    ]
    
    for col in diff_columns:
        if col in dataset.columns:
            dataset[f'{col}_diff'] = dataset[col].diff().fillna(0)
    
    dataset = dataset.fillna(0)
    
    print(f"Dataset created with {len(dataset)} rows and {len(dataset.columns)} features")
    print(f"Attack samples: {dataset['attack'].sum()}")
    print(f"Normal samples: {(dataset['attack'] == 0).sum()}")
    
    return dataset

In [44]:
if __name__ == '__main__':
    # Create dataset
    dataset = create_ml_dataset(
        flow_stats_path,
        port_stats_path,
        table_stats_path,
        alerts_path
    )
    
    # Save dataset
    dataset.to_csv(output_path, index=False)
    print(f"\nDataset saved to {output_path}")

Parsing timestamps...
Alerts before removing NaT: 114930
Alerts after removing NaT: 114930

Filtering data to time range: 2025-10-14 19:14:18 to 2025-10-14 20:15:09
Rows after filtering - Flow: 2177535, Port: 17835, Table: 11876, Alerts: 114930

Sample flow timestamp: 2025-10-14 19:14:18
Sample flow time_bucket: 2025-10-14 19:14:18
Sample alert timestamp: 2025-10-14 19:14:18.570819
Sample alert time_bucket: 2025-10-14 19:14:18
Total alerts: 114930

Checking timestamp overlap...
Flow time range: 2025-10-14 19:14:18 to 2025-10-14 20:15:09
Alert time range: 2025-10-14 19:14:18 to 2025-10-14 20:12:31

First 5 alert time buckets:
[Timestamp('2025-10-14 19:14:18'), Timestamp('2025-10-14 19:14:18'), Timestamp('2025-10-14 19:14:18'), Timestamp('2025-10-14 19:14:18'), Timestamp('2025-10-14 19:14:18')]

First 5 flow time buckets:
[Timestamp('2025-10-14 19:14:18'), Timestamp('2025-10-14 19:14:18'), Timestamp('2025-10-14 19:14:18'), Timestamp('2025-10-14 19:14:18'), Timestamp('2025-10-14 19:14:19'

In [45]:
# Display sample and statistics
print("\nDataset Info:")
print(dataset.info())
print("\nFirst few rows:")
print(dataset.head())
print("\nFeature statistics:")
print(dataset.describe())
print("\nClass distribution:")
print(dataset['attack'].value_counts())


Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3414 entries, 0 to 3413
Data columns (total 69 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   timestamp                        3414 non-null   datetime64[ns]
 1   flow_packet_count_sum            3414 non-null   int64         
 2   flow_packet_count_mean           3414 non-null   float64       
 3   flow_packet_count_std            3414 non-null   float64       
 4   flow_byte_count_sum              3414 non-null   int64         
 5   flow_byte_count_mean             3414 non-null   float64       
 6   flow_byte_count_std              3414 non-null   float64       
 7   flow_duration_sec_mean           3414 non-null   float64       
 8   flow_duration_sec_std            3414 non-null   float64       
 9   port_rx_packets_sum              3414 non-null   int64         
 10  port_rx_packets_mean             3414 non-nul

In [46]:
# Check attack distribution
print("\nAttack distribution:")
print(dataset['attack'].value_counts())

# Show some attack samples
attack_samples = dataset[dataset['attack'] == 1]
print(f"\nAttack samples: {len(attack_samples)}")
if len(attack_samples) > 0:
    print("\nFirst 10 attack rows:")
    print(attack_samples[['timestamp', 'no_of_total_flows', 'no_of_tcp_flows', 
                          'no_of_unique_src_addresses', 'attack']].head(10))

# Check flow distribution over time
print("\nFlow count distribution:")
print(dataset['no_of_total_flows'].describe())

# Show rows with high flow counts
high_flow = dataset[dataset['no_of_total_flows'] > 100]
print(f"\nRows with >100 flows: {len(high_flow)}")
print(high_flow[['timestamp', 'no_of_total_flows', 'no_of_tcp_flows', 'attack']].head(10))


Attack distribution:
attack
0    2656
1     758
Name: count, dtype: int64

Attack samples: 758

First 10 attack rows:
            timestamp  no_of_total_flows  no_of_tcp_flows  \
0 2025-10-14 19:14:18                  4                0   
1 2025-10-14 19:14:19                  4                0   
2 2025-10-14 19:14:20                  4                0   
3 2025-10-14 19:14:22                318              316   
4 2025-10-14 19:14:23                393              391   
5 2025-10-14 19:14:24                942              940   
6 2025-10-14 19:14:26               1572             1570   
7 2025-10-14 19:14:27               2802             2798   
8 2025-10-14 19:14:28               3741             3739   
9 2025-10-14 19:14:29               5904             5898   

   no_of_unique_src_addresses  attack  
0                           0       1  
1                           0       1  
2                           0       1  
3                           2       1  
4        

In [47]:
# Check if features are diverse
print("\nChecking feature diversity:")
print(f"Unique no_of_total_flows values: {dataset['no_of_total_flows'].nunique()}")
print(f"Unique no_of_tcp_flows values: {dataset['no_of_tcp_flows'].nunique()}")
print(f"Unique no_of_unique_src_addresses values: {dataset['no_of_unique_src_addresses'].nunique()}")

# Check packet/byte counts
print("\nPacket and byte statistics:")
print(dataset[['flow_packet_count_sum', 'flow_byte_count_sum', 
               'port_rx_packets_sum', 'port_tx_packets_sum']].describe())

# Compare attack vs normal
print("\nAttack vs Normal comparison:")
print("\nNormal samples (mean):")
print(dataset[dataset['attack']==0][['no_of_total_flows', 'no_of_tcp_flows', 
                                      'flow_packet_count_sum']].mean())
print("\nAttack samples (mean):")
print(dataset[dataset['attack']==1][['no_of_total_flows', 'no_of_tcp_flows', 
                                      'flow_packet_count_sum']].mean())


Checking feature diversity:
Unique no_of_total_flows values: 1036
Unique no_of_tcp_flows values: 1030
Unique no_of_unique_src_addresses values: 521

Packet and byte statistics:
       flow_packet_count_sum  flow_byte_count_sum  port_rx_packets_sum  \
count           3.414000e+03         3.414000e+03         3.414000e+03   
mean            2.018931e+06         1.428192e+08         1.170984e+06   
std             2.208976e+06         1.534237e+08         1.260783e+06   
min             0.000000e+00         0.000000e+00         0.000000e+00   
25%             3.070030e+05         2.266592e+07         1.693545e+05   
50%             2.147592e+06         1.355107e+08         1.173203e+06   
75%             3.480100e+06         2.302908e+08         1.989028e+06   
max             6.220729e+07         4.538778e+09         3.482108e+07   

       port_tx_packets_sum  
count         3.414000e+03  
mean          4.517402e+05  
std           3.753399e+05  
min           0.000000e+00  
25%       